# 6.3 — Dense and Sparse Vectors

**Chapter 6, section 6.2.1**, and the method behind **Exercise 3**.

**The question this notebook answers:** when does the sparse representation of a vector or matrix
matter? The honest answer is arithmetic, and this notebook does the arithmetic rather than
asserting the conclusion.

Nothing here is distributed. These are *local* types: they live wherever the Python code that
creates them runs, whether that is the driver or the inside of a task. The distributed matrices
built out of them are notebook 6.4.

Runs on a laptop in seconds. No Spark session is needed for most of it, but one is opened so the
MLlib types are the ones a student will actually meet.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
import numpy as np
import scipy.sparse as sps
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-6.3")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

import scipy
print("Spark", spark.version, "| NumPy", np.__version__, "| SciPy", scipy.__version__)

Spark 4.2.0 | NumPy 2.5.2 | SciPy 1.18.1


---

## 1. One vector, four constructions

A local vector has integer, 0-based indices and double values. MLlib supports two
representations, and is deliberately permissive about what counts as each.

* **dense** — one array of doubles holding **every** entry, zeros included.
* **sparse** — two parallel arrays, one of indices and one of values, holding **only the
  non-zeros**.

The documentation's worked example is the clearest statement of the difference: the vector
$(1.0,\ 0.0,\ 3.0)$ is `[1.0, 0.0, 3.0]` dense and `(3, [0, 2], [1.0, 3.0])` sparse, where `3` is
the size, `[0, 2]` the positions of the non-zeros and `[1.0, 3.0]` their values.

In [2]:
from pyspark.ml.linalg import Vectors

# Use a NumPy array as a dense vector.
dv1 = np.array([1.0, 0.0, 3.0])
# Use a Python list as a dense vector.
dv2 = [1.0, 0.0, 3.0]
# Create a SparseVector.
sv1 = Vectors.sparse(3, [0, 2], [1.0, 3.0])
# Use a single-column SciPy csc_matrix as a sparse vector.
sv2 = sps.csc_matrix((np.array([1.0, 3.0]), np.array([0, 2]),
                      np.array([0, 2])), shape=(3, 1))

print("dv1", type(dv1).__name__, dv1)
print("dv2", type(dv2).__name__, dv2)
print("sv1", type(sv1).__name__, sv1)
print("sv2", type(sv2).__name__, "\n", sv2.toarray().ravel())

dv1 ndarray [1. 0. 3.]
dv2 list [1.0, 0.0, 3.0]
sv1 SparseVector (3,[0,2],[1.0,3.0])
sv2 csc_matrix 
 [1. 0. 3.]


Four constructions, one mathematical object: two dense forms differing only in container, and two
sparse forms, one from MLlib's own factory and one from SciPy. The `Vectors.sparse` call reads
directly off the encoding — size, index array, value array.

They are the same vector, and saying so as an assertion is better than saying so in a sentence.

In [3]:
target = np.array([1.0, 0.0, 3.0])
for name, v in [("dv1", dv1), ("dv2", np.array(dv2)),
                ("sv1", sv1.toArray()), ("sv2", sv2.toarray().ravel())]:
    assert np.allclose(v, target), name
    print(f"{name}: {v}  -> same vector")

dv1: [1. 0. 3.]  -> same vector
dv2: [1. 0. 3.]  -> same vector
sv1: [1. 0. 3.]  -> same vector
sv2: [1. 0. 3.]  -> same vector


### A version drift worth recording

Spark's documentation names SciPy's `csc_matrix`, with a single column, as an accepted
sparse-vector type. SciPy's current documentation instead documents `csc_array`, and states that
the library is shifting from its sparse *matrix* interface to a sparse *array* interface and
expects to deprecate the matrix interface in coming releases.

Both statements are correct as published; they are published by two projects moving at different
speeds. This is what to expect wherever two independently maintained libraries meet, and it is the
reason to check the documentation of **both** sides of an interface rather than either alone.

In [4]:
sv2_array = sps.csc_array((np.array([1.0, 3.0]), np.array([0, 2]),
                           np.array([0, 2])), shape=(3, 1))
print("csc_matrix ->", type(sv2).__name__)
print("csc_array  ->", type(sv2_array).__name__)
print("same values:", np.allclose(sv2.toarray(), sv2_array.toarray()))

csc_matrix -> csc_matrix
csc_array  -> csc_array
same values: True


---

## 2. What each representation costs

A dense vector pays 8 bytes per position, zero or not. A sparse vector pays an index and a value
per non-zero — 4 bytes plus 8 bytes in PySpark's encoding, where the index array is `int32`.

The cell below reads those sizes off the objects themselves rather than assuming them.

In [5]:
from pyspark.ml.linalg import SparseVector, DenseVector

def nbytes(v):
    # the payload arrays only: what the numbers themselves occupy
    if isinstance(v, SparseVector):
        return v.indices.nbytes + v.values.nbytes
    return np.asarray(v).nbytes

print("index dtype :", sv1.indices.dtype, "->", sv1.indices.itemsize, "bytes each")
print("value dtype :", sv1.values.dtype, "->", sv1.values.itemsize, "bytes each")
print()
print("dense  (3 entries, 2 non-zero):", nbytes(DenseVector([1.0, 0.0, 3.0])), "bytes")
print("sparse (3 entries, 2 non-zero):", nbytes(sv1), "bytes")

index dtype : int32 -> 4 bytes each
value dtype : float64 -> 8 bytes each

dense  (3 entries, 2 non-zero): 24 bytes
sparse (3 entries, 2 non-zero): 24 bytes


On a three-element vector with two non-zeros the sparse form saves **nothing at all**: 24 bytes
either way. Two non-zeros in three entries is a density of $2/3$, which is exactly the crossover
computed in the next section, so the tie is not a coincidence. The saving appears when the
fraction of non-zeros is small and the vector is long — the normal condition of text and of
one-hot encoded categorical features.

**One-hot encoding** expands a categorical value into a vector of indicator entries, one per
category, all but one of them zero. It manufactures exactly the wide, mostly-zero vectors the
sparse form is built for.

In [6]:
n_categories = 100_000
category = 42_317

dense_one_hot = np.zeros(n_categories)
dense_one_hot[category] = 1.0
sparse_one_hot = Vectors.sparse(n_categories, [category], [1.0])

d, s = nbytes(dense_one_hot), nbytes(sparse_one_hot)
print(f"one-hot over {n_categories:,} categories")
print(f"  dense : {d:>9,} bytes")
print(f"  sparse: {s:>9,} bytes   ({d / s:,.0f}x smaller)")
assert np.allclose(sparse_one_hot.toArray(), dense_one_hot)

one-hot over 100,000 categories
  dense :   800,000 bytes
  sparse:        12 bytes   (66,667x smaller)


---

## 3. The crossover, computed

Sparse storage costs $12$ bytes per non-zero where dense costs $8$ bytes per entry, so sparse
costs about **one and a half dense entries per non-zero**. It therefore wins whenever fewer than
roughly **two thirds** of the entries are non-zero.

That is a claim with a number in it, so it gets computed rather than quoted.

In [7]:
BYTES_DENSE_ENTRY = 8      # one float64
BYTES_SPARSE_NNZ = 12      # one int32 index + one float64 value

size = 1_000
print(f"{'density':>8} {'dense B':>10} {'sparse B':>10}   winner")
for density in [0.001, 0.01, 0.1, 0.5, 0.6, 2/3, 0.7, 1.0]:
    nnz = round(size * density)
    dense_b, sparse_b = size * BYTES_DENSE_ENTRY, nnz * BYTES_SPARSE_NNZ
    winner = "sparse" if sparse_b < dense_b else "dense"
    print(f"{density:>8.3f} {dense_b:>10,} {sparse_b:>10,}   {winner}")

crossover = BYTES_DENSE_ENTRY / BYTES_SPARSE_NNZ
print(f"\ncrossover density = {BYTES_DENSE_ENTRY}/{BYTES_SPARSE_NNZ} = {crossover:.4f}")

 density    dense B   sparse B   winner
   0.001      8,000         12   sparse
   0.010      8,000        120   sparse
   0.100      8,000      1,200   sparse
   0.500      8,000      6,000   sparse
   0.600      8,000      7,200   sparse
   0.667      8,000      8,004   dense
   0.700      8,000      8,400   dense
   1.000      8,000     12,000   dense

crossover density = 8/12 = 0.6667


---

## 4. The case that decides it: a term–document matrix

A **term–document matrix** is the standard representation of a text corpus: each row is a
document, each column a vocabulary term. Take one million documents over a hundred thousand terms.

The point of this cell is that the matrix is never built. Storage is arithmetic on four numbers,
and doing it *before* provisioning anything is the habit the chapter is teaching.

In [8]:
def storage(n_docs, n_terms, terms_per_doc, label):
    entries = n_docs * n_terms
    nnz = n_docs * terms_per_doc
    dense_gb = entries * BYTES_DENSE_ENTRY / 1e9
    sparse_gb = nnz * BYTES_SPARSE_NNZ / 1e9
    print(label)
    print(f"  {n_docs:,} documents x {n_terms:,} terms = {entries:.3g} entries")
    print(f"  {'density':<22}: {nnz / entries:>12.2%}   ({nnz:,} non-zeros)")
    print(f"  {'dense  @ 8 B/entry':<22}: {dense_gb:>12,.1f} GB")
    print(f"  {'sparse @ 12 B/nnz':<22}: {sparse_gb:>12,.1f} GB")
    print(f"  {'ratio':<22}: {dense_gb / sparse_gb:>12,.0f}x")
    return dense_gb, sparse_gb

_ = storage(1_000_000, 100_000, 100, "The chapter's corpus")

The chapter's corpus
  1,000,000 documents x 100,000 terms = 1e+11 entries
  density               :        0.10%   (100,000,000 non-zeros)
  dense  @ 8 B/entry    :        800.0 GB
  sparse @ 12 B/nnz     :          1.2 GB
  ratio                 :          667x


This is the precise sense in which sparse representation is the difference between fitting in
memory and not. Not a constant-factor economy: **the difference between about a gigabyte and
several hundred gigabytes for the same information**, before any working copy a computation makes.

Exercise 3 asks the same question of a different corpus, and the function above is the method.

In [9]:
# Exercise 3's corpus -- the method, applied
_ = storage(500_000, 200_000, 300, "Exercise 3's corpus")

Exercise 3's corpus
  500,000 documents x 200,000 terms = 1e+11 entries
  density               :        0.15%   (150,000,000 non-zeros)
  dense  @ 8 B/entry    :        800.0 GB
  sparse @ 12 B/nnz     :          1.8 GB
  ratio                 :          444x


---

## 5. Local matrices, and the one convention that silently bites

A dense local matrix stores its entries in a single double array in **column-major order** — all
of column 0, then all of column 1, and so on. A sparse local matrix stores its non-zeros in
compressed sparse column (CSC) format, likewise column by column.

The documentation's example makes column-major concrete: the $3\times2$ matrix with rows
$(1,2)$, $(3,4)$, $(5,6)$ is stored as the flat array `[1.0, 3.0, 5.0, 2.0, 4.0, 6.0]`.

In [10]:
from pyspark.ml.linalg import Matrices

# Create a dense matrix ((1.0, 2.0), (3.0, 4.0), (5.0, 6.0))
dm2 = Matrices.dense(3, 2, [1, 3, 5, 2, 4, 6])

# Create a sparse matrix ((9.0, 0.0), (0.0, 8.0), (0.0, 6.0))
sm = Matrices.sparse(3, 2, [0, 1, 3], [0, 2, 1], [9, 6, 8])

print("dense 3x2 from [1, 3, 5, 2, 4, 6]:")
print(dm2.toArray())
print("\nsparse 3x2, CSC (colPtrs, rowIndices, values):")
print(sm.toArray())

dense 3x2 from [1, 3, 5, 2, 4, 6]:
[[1. 2.]
 [3. 4.]
 [5. 6.]]

sparse 3x2, CSC (colPtrs, rowIndices, values):
[[9. 0.]
 [0. 8.]
 [0. 6.]]


Read the dense call against the convention: the flat list `[1, 3, 5, 2, 4, 6]` is the first column
$(1,3,5)$ followed by the second $(2,4,6)$.

A programmer who assumes row-major order constructs **the transpose of the intended matrix, and
no error is raised**, because the transpose is an equally valid matrix. That is the failure to
recognize, so here it is side by side.

In [11]:
intended = np.array([[1., 2.], [3., 4.], [5., 6.]])
column_major = Matrices.dense(3, 2, [1, 3, 5, 2, 4, 6]).toArray()   # correct
row_major_guess = Matrices.dense(3, 2, [1, 2, 3, 4, 5, 6]).toArray()  # the mistake

print("intended:\n", intended)
print("\nwhat the row-major guess builds:\n", row_major_guess)
print("\nis the guess the transpose of a valid matrix? ",
      np.allclose(row_major_guess, np.array([[1., 4.], [2., 5.], [3., 6.]])))
assert np.allclose(column_major, intended)
assert not np.allclose(row_major_guess, intended)
print("\nNo exception was raised by either call.")

intended:
 [[1. 2.]
 [3. 4.]
 [5. 6.]]

what the row-major guess builds:
 [[1. 4.]
 [2. 5.]
 [3. 6.]]

is the guess the transpose of a valid matrix?  True

No exception was raised by either call.


The sparse call's three lists are the CSC encoding — column start offsets, row indices, values —
and are best written by a factory or a conversion rather than by hand. The round trip below shows
what it encodes.

In [12]:
print("colPtrs   :", sm.colPtrs)      # where each column's entries start
print("rowIndices:", sm.rowIndices)   # the row of each non-zero
print("values    :", sm.values)       # the non-zeros themselves
print("\nreconstructed:\n", sm.toArray())

colPtrs   : [0 1 3]
rowIndices: [0 2 1]
values    : [9. 6. 8.]

reconstructed:
 [[9. 0.]
 [0. 8.]
 [0. 6.]]


---

## Conclusion

**Choosing sparse over dense is arithmetic, not preference.** Sparse costs about one and a half
dense entries per non-zero, so it wins below roughly two-thirds density and loses above it. Text
and one-hot encoded features sit at fractions of a percent, which is why they are stored sparsely
without anyone having to argue about it, and why a three-element example makes the sparse form
look pointless.

**Both local matrix formats are column-major**, and assuming otherwise builds a transpose in
silence. Check `toArray()` once after any hand-written `Matrices.dense`.

These types live on one machine. Notebook 6.4 takes the same question to matrices that do not.

In [13]:
spark.stop()